# PowerIt (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/PowerIt_java.ipynb)

Capitolo 18 — Algoritmi ricorsivi e iterativi a confronto. Notebook eseguibile di `PowerIt`: l'algoritmo in Java, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/PowerIt_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/PowerIt_cpp.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_java.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. La prima cella verifica che il JDK ci sia e, se manca, lo installa. Ogni programma Java è scritto su un file dalla cella `%%writefile`, poi compilato con `javac` ed eseguito con `java` nella cella successiva.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `fibRec` (camelCase).

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Potenza veloce: ricorsiva e PowerIt (square-and-multiply)

- **FastPower**$(x,n)$ (capitolo 14) — ricorsiva: $x^n=(x^{\lfloor n/2\rfloor})^2$ se $n$ è pari, $(x^{\lfloor n/2\rfloor})^2\cdot x$ se è dispari. $\Theta(\log_2 n)$ passi, $\lfloor\log_2 n\rfloor+2$ chiamate (profondità $\Theta(\log_2 n)$). Il risultato della chiamata ricorsiva è salvato in $y$ e usato due volte.
- **PowerIt**$(x,n)$ — *Input*: un numero $x$ e un naturale $n$. *Output*: $x^n$. Legge i bit di $n$ dal meno significativo: $\lfloor\log_2 n\rfloor+1$ iterazioni, memoria extra $\Theta(1)$; vale $p\cdot y^k=x^n$ a ogni iterazione.

Con $x=2$ e $n=13=(1101)_2$ si stampa la traccia del capitolo: $p$ viene moltiplicato per $x$, $x^4$ e $x^8$ (i bit di 13 uguali a 1) e $p=x^{13}$ dopo $4$ iterazioni. Poi la tabella: stesso risultato, $\lfloor\log_2 n\rfloor+1$ iterazioni contro $\lfloor\log_2 n\rfloor+2$ chiamate, contro le $n$ moltiplicazioni del ciclo naïf.

In [2]:
%%writefile Power.java
import java.util.*;

public class Power {
    static int powCalls = 0;   // chiamate di FastPower
    static int powIters = 0;   // iterazioni di PowerIt

    // fastPower(x, n): x^n per ricorsione sul dimezzamento di n
    static long fastPower(long x, int n) {
        powCalls++;
        if (n == 0) {
            return 1;                   // caso base
        }
        long y = fastPower(x, n / 2);   // una sola chiamata ricorsiva
        if (n % 2 == 0) {
            return y * y;               // n pari
        }
        return x * y * y;               // n dispari
    }

    // powerIt(x, n): x^n con square-and-multiply
    static long powerIt(long x, int n) {
        long p = 1;
        long y = x;
        int k = n;
        while (k > 0) {
            powIters++;
            if (k % 2 == 1) {
                p = p * y;
            }
            y = y * y;
            k = k / 2;
        }
        return p;
    }

    static void powerItTrace(long x, int n) {
        System.out.println("PowerIt(" + x + ", " + n + "): " + n + " = (" + Integer.toBinaryString(n) + ")_2");
        long p = 1, y = x;
        int k = n, it = 0;
        while (k > 0) {
            it++;
            String par = k % 2 == 1 ? "dispari" : "pari";
            if (k % 2 == 1) {
                p = p * y;
            }
            y = y * y;
            System.out.println(it + ". k=" + k + " " + par + ": p=" + p + ", y=" + y + ", k=" + (k / 2));
            k = k / 2;
        }
        System.out.println("iterazioni = " + it + ", risultato p = " + p);
    }

    public static void main(String[] args) {
        powerItTrace(2, 13);
        System.out.println();
        System.out.printf("%3s %12s %8s %10s %6s   (iterazioni, chiamate, moltiplicazioni)%n", "n", "2^n", "PowerIt", "FastPower", "naive");
        int[] ns = {0, 1, 2, 5, 13, 30};
        for (int n : ns) {
            powIters = 0;
            long r1 = powerIt(2, n);
            powCalls = 0;
            long r2 = fastPower(2, n);
            System.out.printf("%3d %12d %8d %10d %6d   %s%n", n, r1, powIters, powCalls, n, (r1 == r2 && r1 == (1L << n)) ? "OK" : "ERROR");
        }
    }
}

Writing Power.java


In [3]:
!javac Power.java && java Power

PowerIt(2, 13): 13 = (1101)_2
1. k=13 dispari: p=2, y=4, k=6
2. k=6 pari: p=2, y=16, k=3
3. k=3 dispari: p=32, y=256, k=1
4. k=1 dispari: p=8192, y=65536, k=0
iterazioni = 4, risultato p = 8192

  n          2^n  PowerIt  FastPower  naive   (iterazioni, chiamate, moltiplicazioni)
  0            1        0          1      0   OK
  1            2        1          2      1   OK
  2            4        2          3      2   OK
  5           32        3          4      5   OK
 13         8192        4          5     13   OK
 30   1073741824        5          6     30   OK


**Prova tu.** Prova un'altra base e un altro esponente (in C++ e Java il risultato deve stare in 64 bit).

In [4]:
%%writefile ProvaPower.java
import java.util.*;

public class ProvaPower {
    public static void main(String[] args) {
        // Prova un'altra base e un altro esponente (attenzione: in C++ e Java il risultato deve stare in 64 bit).
        Power.powIters = 0;
        Power.powCalls = 0;
        long r1 = Power.powerIt(3, 25);
        long r2 = Power.fastPower(3, 25);
        System.out.println(r1 + " " + Power.powIters + " " + r2 + " " + Power.powCalls);
    }
}

Writing ProvaPower.java


In [5]:
!javac ProvaPower.java && java ProvaPower

847288609443 5 847288609443 6


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).